# Notebook 4 of 6 — III.b Classification benchmark and optimisation
**Group XX** · Twelve algorithms from class under one protocol, random-search tuning of six families, a soft-voting check, the decision threshold, and one single look at the sealed holdout.

| | From Part 1 (`notebooks/04_Model.ipynb`) | In Part 2 |
|---|---|---|
| Validation | 4 chronological windows (last 4 of 10 date blocks), 90-day gap | **Kept** the chronological windows and the gap; **changed** to 5 yearly windows plus a **sealed holdout**, because Part 1 used the same windows to choose `C` and to report the score |
| Pipeline | Preprocessing + selector + model refitted in each window (week 5) | **Kept** |
| Models | Logistic regression, `C` in {0.001 … 100} | **Kept** logistic regression and that `C` range inside its search; **new:** 11 more algorithms from weeks 6–13, random search, soft voting |
| Metric | F1 at threshold 0.5 | **Changed:** ROC-AUC to compare models, then F1 at a tuned threshold, always shown next to the F1 of "always repeat" (Part 1's 0.76 was close to that 0.72 floor) |
| Final fit | Refit on all labelled rows, predict `test.csv` | **Kept** (without the empty rows) |

In [ ]:
import json
import tempfile
import time
import warnings
from pathlib import Path
import sys
import os

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm
from joblib import Memory
from sklearn.base import clone
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier, VotingClassifier)
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier


project_dir = os.path.abspath("..")

# Add it to Python's import path
sys.path.insert(0, project_dir)

import helpers.project_pipeline as pp
import importlib
importlib.reload(pp)

warnings.filterwarnings("ignore", message="Found unknown categories")
# Some sampled MLP settings stop at their iteration limit; that is part of what the search compares.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.width", 170)
BLUE, ORANGE, GREY = "#1779A8", "#C96A00", "#8C8C8C"
R = pp.RANDOM_STATE
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)

train, test, _ = pp.load_raw(pp.find_data_folder())
design = pp.development_split(train)
X, y, folds = design.X_dev, design.y_dev, design.folds

cache_folder = tempfile.TemporaryDirectory()
# Week 11: Pipeline(memory=...) caches the fitted preprocessing + selection of each fold, so the
# 60 candidates of a search do not refit them again. Results are identical, only faster.
memory = Memory(cache_folder.name, verbose=0)

def pipeline(model, **prep_options):
    """The project pipeline: preprocessing -> vote feature selection (notebook 03) -> model."""
    return pp.make_pipeline(model, selector=pp.make_vote_selector(), memory=memory, **prep_options)

## 4.1 Protocol and metrics

- **Rows:** the 5,564 development purchases; the five yearly windows of notebook 01 (each trains on purchases more than 90 days before it opens). The sealed holdout is used once, in Section 4.8.
- **Pipeline:** every model gets the same preprocessing (notebook 02) and vote selection (notebook 03), refitted in each window.
- **Comparison metric: ROC-AUC.** It does not depend on a threshold, so it compares how well models *rank* purchases. F1 at 0.5 and balanced accuracy at 0.5 are shown beside it, but F1 depends heavily on the threshold, which is chosen afterwards for the final model (Section 4.7).
- **Uncertainty:** mean and standard deviation over the five windows; differences between models are paired by window (a difference inside 2 × SEM is "inside the noise").
- **Selection rule, fixed in advance:** the final model is the candidate with the highest mean validation ROC-AUC; an ensemble is preferred over a single model only if it is better beyond the noise.

## 4.2 Baselines: what any model must beat


In [ ]:
def baseline_scores(rule):
    rows = []
    for number, (_, score_at) in enumerate(folds, start=1):
        predicted = rule(X.iloc[score_at]).astype(int)
        s = pp.scores(y.iloc[score_at], predicted, threshold=0.5)
        rows.append({"fold": number, **s})
    return pd.DataFrame(rows).set_index("fold")

baselines = {
    "always 'repeat'": baseline_scores(lambda rows: np.ones(len(rows))),
    "rule: bought in the last 90 days": baseline_scores(lambda rows: rows["purchase_count_90d"].fillna(0) > 0),
}
display(pd.DataFrame({name: table[["ROC_AUC", "F1", "balanced_acc"]].mean() for name, table in baselines.items()}).T.round(4))

Predicting "repeat" for everyone already gives F1 = 0.703 on these windows, with no ranking ability (AUC 0.5) and balanced accuracy 0.5. A sensible business rule ("customers who bought in the last 90 days will buy again") separates the classes somewhat (balanced accuracy 0.66) but scores a *lower* F1 (0.69) than predicting "repeat" for everyone. These are the reference points for every model below.

## 4.3 Benchmark: twelve algorithms from class with default settings

One representative of every family covered in class: linear (logistic regression, LDA), Bayesian (Gaussian naive Bayes), distance-based (KNN), a single tree, bagging (bagged trees, random forest, extra trees), boosting (AdaBoost, gradient boosting, histogram gradient boosting) and a neural network (MLP). Library defaults are used, except a larger iteration limit for logistic regression and the MLP so that they finish fitting.

In [ ]:
# One algorithm per family seen in class, with library defaults (only a larger max_iter so that
# the iterative fits finish): week 6 linear, week 7 naive Bayes, week 8 KNN, week 9 tree,
# week 10 bagging / forests, week 11 boosting, week 12 MLP.
defaults = {
    ### STANDALONE MODELS
    "LogisticRegression": LogisticRegression(max_iter=5000, random_state=R),
    "LDA": LinearDiscriminantAnalysis(),
    "GaussianNB": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=R),
    "ExtraTrees": ExtraTreesClassifier(random_state=R, n_jobs=1),
    "MLP": MLPClassifier(max_iter=1000, random_state=R),

    ### ENSEMBLES
    "Bagging": BaggingClassifier(random_state=R),
    "AdaBoost": AdaBoostClassifier(random_state=R),
    "GradientBoosting": GradientBoostingClassifier(random_state=R),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=R),
    "RandomForest": RandomForestClassifier(random_state=R, n_jobs=1),
}

default_scores, rows = {}, []

for name, model in tqdm(defaults.items(), total=len(defaults), desc="Benchmarking models", unit="model"):
    start = time.time()

    default_scores[name], _ = pp.cross_validate_time(
        pipeline(model), X, y, folds
    )

    elapsed = round(time.time() - start, 1)

    rows.append({
        "model": name,
        **pp.summarise(default_scores[name]),
        "seconds (5 windows)": elapsed
    })

    tqdm.write(
        f"{name} completed | "
        f"ROC-AUC: {default_scores[name]['ROC_AUC'].mean():.4f} | "
        f"Time: {elapsed}s"
    )

benchmark = pd.DataFrame(rows).set_index("model").sort_values("ROC_AUC mean", ascending=False)

display(benchmark.round(4))


In [ ]:
# ROC-AUC of every model in every window (one dot per window).
order = benchmark.index
fig, ax = plt.subplots(figsize=(10, 4.2))
for i, name in enumerate(order):
    values = default_scores[name]["ROC_AUC"]
    ax.scatter(np.full(len(values), i), values, color=BLUE, alpha=0.6)
    ax.scatter(i, values.mean(), color=ORANGE, marker="D", s=50, zorder=3)
ax.axhline(0.5, color=GREY, linestyle=":")
ax.set_xticks(range(len(order)), order, rotation=35, ha="right")
ax.set(ylabel="validation ROC-AUC", title="Default settings: each dot is one yearly window, diamond = mean")
plt.tight_layout()
plt.show()

- With default settings the **linear models rank first** (logistic regression 0.790, LDA 0.784), just ahead of extra trees (0.780) and random forest (0.779), then the boosting models (0.765–0.771).
- Models that memorise their training rows lose most: the unpruned **decision tree** (0.614) and **bagging** of ten unpruned trees (0.726). **KNN** with 5 neighbours (0.734) and the default **MLP** (0.725) are also weak, as weeks 8 and 12 predict for a small *k* and an unregularised network.
- **Gaussian naive Bayes** ranks reasonably (0.743) but its F1 at 0.5 collapses (0.51 ± 0.17): its probabilities are extreme because the many correlated history columns break its independence assumption (week 7).
- Every model except the single tree ranks purchases far better than the business rule, and the spread between windows (std 0.02–0.03) is as large as the gaps between the best models: the yearly windows differ more from each other than the top algorithms do. Paired comparisons are therefore needed.

## 4.4 Optimisation: random search over six families


The search tunes one model of each kind that can still improve: logistic regression (linear), KNN (distance), random forest and extra trees (bagging), histogram gradient boosting (boosting) and the MLP (neural). KNN and the MLP are included despite weak defaults because their defaults are known to overfit (5 neighbours, an unregularised 100-unit layer), so the default ranking understates them.

As in class, each search draws **60 candidates** at random, without replacement, from the listed values: with 60 draws there is about a 95 % chance of including at least one candidate from the best 5 % of the space, whatever its size (Bergstra & Bengio, 2012). Every candidate is scored on the same five windows with the whole pipeline refitted inside each, so selection and preprocessing never see validation rows. The rare-level merging of notebook 02 (`min_frequency`) is searched together with the model settings.

In [ ]:

merge = {
    "prep__columns__cat__encode__min_frequency": [1, 2, 3, 5]
}


search_spaces = {
    "LogisticRegression": (pp.base_model("LogisticRegression"), {
        "model__C": [0.01, 0.02, 0.03, 0.05],
        "model__l1_ratio": [0, 0.05, 0.1],
        "model__class_weight": [None, "balanced"],
        **merge
    }),

    "KNN": (pp.base_model("KNN"), {
        "model__n_neighbors": [75, 101, 125, 151, 175],
        "model__weights": ["distance", "uniform"],
        "model__p": [1, 2],
        **merge
    }),

    "RandomForest": (pp.base_model("RandomForest"), {
        "model__n_estimators": [400, 600, 800, 1000],
        "model__max_depth": [None, 18, 25],
        "model__min_samples_leaf": [4, 6, 8, 12],
        "model__max_features": ["sqrt", "log2", 0.2],
        "model__criterion": ["gini", "entropy"],
        "model__class_weight": ["balanced", "balanced_subsample"],
        **merge
    }),

    "ExtraTrees": (pp.base_model("ExtraTrees"), {
        "model__n_estimators": [500],
        "model__min_samples_leaf": [10],
        "model__max_features": [0.3],
        "model__max_depth": [16],
        "model__criterion": ["entropy"],
        "model__class_weight": ["balanced_subsample"],
        **merge
    }),

    "AdaBoost": (pp.base_model("AdaBoost"), {
        "model__n_estimators": [300, 500, 700, 900],
        "model__learning_rate": [0.05, 0.08, 0.1, 0.15, 0.2],
        **merge
    }),

    "HistGradientBoosting": (pp.base_model("HistGradientBoosting"), {
        "model__learning_rate": [0.02, 0.03, 0.05, 0.08],
        "model__max_iter": [200, 300, 500, 700],
        "model__max_leaf_nodes": [5, 7, 10, 15],
        "model__max_depth": [3, 5, 7],
        "model__min_samples_leaf": [40, 50, 75, 100],
        "model__l2_regularization": [3, 10, 20, 50, 75],
        **merge
    }),

    "MLP": (pp.base_model("MLP"), {
        "model__hidden_layer_sizes": [
            (16,), (32,), (48,), (64,), (96,),
            (32, 16), (64, 32), (64, 32, 16)
        ],
        "model__activation": ["relu", "tanh"],
        "model__alpha": [0.1, 0.3, 1, 3, 10],
        "model__learning_rate_init": [
            0.0001, 0.0003, 0.0005, 0.001
        ],
        "model__batch_size": [64, 100, 200, 256],
        "model__learning_rate": ["constant", "adaptive"],
        "model__early_stopping": [False],
        **merge
    }),
}

# Number combinations for hyperparameter optimization
print({
    name: int(np.prod([len(v) for v in space.values()]))
    for name, (_, space) in search_spaces.items()
})

In [ ]:

import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import RandomizedSearchCV

searches, rows = {}, []

for name, (model, space) in tqdm(search_spaces.items(), total=len(search_spaces), desc="Tuning models", unit="model"):
    start = time.time()
    
    trials = 120
    
    # Run 80 trials for each
    n_candidates = min(trials, int(np.prod([len(v) for v in space.values()])))

    search = RandomizedSearchCV(
        pipeline(model),
        space,
        n_iter=n_candidates,
        cv=folds,
        scoring="roc_auc",
        refit=False,
        n_jobs=-1,
        random_state=R
    )

    search.fit(X, y)
    searches[name] = search

    elapsed = round(time.time() - start, 1)

    rows.append({
        "model": name,
        "candidates": n_candidates,
        "best ROC-AUC": search.best_score_,
        "default ROC-AUC": default_scores[name]["ROC_AUC"].mean(),
        "seconds": elapsed
    })

    tqdm.write(
        f"{name} completed | "
        f"Best ROC-AUC: {search.best_score_:.4f} | "
        f"Trials: {n_candidates} | "
        f"Time: {elapsed}s"
    )


# Display tuning summary
tuning = pd.DataFrame(rows).set_index("model")

tuning["gain from tuning"] = (
    tuning["best ROC-AUC"] - tuning["default ROC-AUC"]
)

display(
    tuning.round(4).sort_values(
        "best ROC-AUC", ascending=False
    )
)

# Display best parameters
for name, search in searches.items():
    print(
        f"{name:22s}",
        {
            k.replace("model__", "").replace(
                "prep__columns__cat__encode__", ""
            ): v
            for k, v in search.best_params_.items()
        }
    )


In [ ]:
def tuned_pipeline(name):
    model, _ = search_spaces[name]
    return pipeline(clone(model)).set_params(**searches[name].best_params_)

# Re-score each tuned pipeline on the five windows to get its per-window scores and
# out-of-fold probabilities (identical folds and seeds, so the AUC matches the search).
tuned_scores, tuned_oof = {}, {}
for name in search_spaces:
    tuned_scores[name], tuned_oof[name] = pp.cross_validate_time(tuned_pipeline(name), X, y, folds)
effect = pd.DataFrame({name: pp.paired_delta(tuned_scores[name]["ROC_AUC"], default_scores[name]["ROC_AUC"])
                       for name in search_spaces}).T
display(effect.round(4))

- Tuning improves every family beyond the noise on all five windows, except **logistic regression**, whose default is already close to its best (`C = 0.03`, +0.003, inside the noise).
- The largest gains are for the models whose defaults overfit: **MLP +0.070** (one small layer of 8 units with a strong L2 penalty, `alpha = 3`), **KNN +0.054** (101 neighbours, distance-weighted, Manhattan distance), **histogram boosting +0.025** (7-leaf trees, slow learning rate, strong L2), **extra trees +0.020** and **random forest +0.014** (leaves of at least 5 purchases).
- The common thread is **strong regularisation**: the signal is modest and the yearly windows shift, so the settings that average more and fit less generalise best.
- The rare-level setting chosen is 1 or 10, never 30, which agrees with notebook 02.
- Each best score is the best of 60 candidates on the same windows, so it is slightly optimistic; the sealed holdout (4.8) corrects for this.

## 4.5 Does combining the strongest models help? (soft voting)


Soft voting averages the probabilities of several tuned pipelines (week 11). It can only help if the members make different mistakes. The three best tuned models are combined, and the combination must beat the best single model beyond the noise to be chosen.

In [ ]:
ranking = pd.Series({name: tuned_scores[name]["ROC_AUC"].mean() for name in tuned_scores}).sort_values(ascending=False)
best_single = ranking.index[0]
top3 = list(ranking.index[:3])
vote_model = VotingClassifier([(name, tuned_pipeline(name)) for name in top3], voting="soft")
vote_scores, vote_oof = pp.cross_validate_time(vote_model, X, y, folds)
print("Members:", top3)
print("Soft vote ROC-AUC:", round(vote_scores["ROC_AUC"].mean(), 4), "| best single:", best_single, round(ranking.iloc[0], 4))
print("Vote vs best single:", {k: (round(v, 4) if isinstance(v, float) else v)
                              for k, v in pp.paired_delta(vote_scores["ROC_AUC"], tuned_scores[best_single]["ROC_AUC"]).items()})

The soft vote of extra trees, MLP and logistic regression reaches 0.7984, slightly *below* extra trees alone (−0.0012, inside the noise, ahead in 2 of 5 windows). The three members rely on the same customer-history signal and make similar mistakes, so averaging has little to cancel. Following the rule fixed in 4.1 (and the advice of weeks 6 and 10 to prefer the simpler model when a more complex one only matches it), the single model is kept.

## 4.6 Comparison of the candidates and choice of the final model


In [ ]:
candidates = {**{f"{name} (tuned)": tuned_scores[name] for name in tuned_scores},
              f"soft vote ({' + '.join(top3)})": vote_scores}
rows = []
reference = tuned_scores[best_single]["ROC_AUC"]
for name, table in candidates.items():
    delta = pp.paired_delta(table["ROC_AUC"], reference)
    rows.append({"candidate": name, **pp.summarise(table), "AUC vs best": delta["mean_diff"],
                 "SEM": delta["SEM"], "wins": delta["wins"], "verdict": delta["verdict"]})
comparison = pd.DataFrame(rows).set_index("candidate").sort_values("ROC_AUC mean", ascending=False)
display(comparison.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
for name, table in candidates.items():
    ax.plot(table.index, table["ROC_AUC"], marker="o", label=name,
            linewidth=2.5 if name.startswith(best_single) else 1, alpha=1 if name.startswith(best_single) else 0.6)
ax.set(xlabel="validation window (yearly, oldest to newest)", ylabel="ROC-AUC", xticks=range(1, 6),
       title="Tuned candidates window by window")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

- **Tuned extra trees** has the highest mean validation ROC-AUC (0.800), is ahead of every other single model in at least 4 of the 5 windows and has the **smallest spread across windows** (std 0.023).
- The other candidates are within 0.005–0.011 AUC; only histogram boosting is behind beyond the noise. The data, not the algorithm, sets the ceiling at about 0.80 AUC.
- All candidates move together from window to window (window 2 is the hardest for everyone): this is the yearly drift seen in notebook 01, not a property of one model.

**Final model: tuned `ExtraTreesClassifier`** — 300 randomised trees grown without a depth limit but with at least 5 purchases per leaf, √p candidate columns per split, entropy criterion, levels seen fewer than 10 times merged, vote feature selection in front. Tuned logistic regression is the natural simple fallback (−0.007, inside the noise), but it is behind in 4 of 5 windows.

## 4.7 Choosing the decision threshold


The model outputs a probability; the label needs a threshold (week 5). F1 rewards finding the repeat purchases, which are the majority, so its best threshold is usually below 0.5. The threshold is chosen on the **out-of-fold probabilities of the five windows** (no holdout row is used).

In [ ]:
final_name = best_single
final_oof = tuned_oof[final_name]
scored = final_oof.notna()
curve = pp.f1_by_threshold(y[scored], final_oof[scored])
threshold = float(curve.idxmax())
per_window = [float(pp.f1_by_threshold(y.iloc[b], final_oof.iloc[b]).idxmax()) for _, b in folds]
print(f"Best pooled threshold: {threshold:.2f} (F1 {curve.max():.4f} vs {curve.loc[0.5]:.4f} at 0.50)")
print("Best threshold in each window:", per_window)

window_f1 = pd.DataFrame({
    "F1 at 0.50": [pp.scores(y.iloc[b], final_oof.iloc[b], 0.5)["F1"] for _, b in folds],
    f"F1 at {threshold:.2f}": [pp.scores(y.iloc[b], final_oof.iloc[b], threshold)["F1"] for _, b in folds],
    "F1 floor": [pp.scores(y.iloc[b], final_oof.iloc[b])["F1_floor"] for _, b in folds],
    f"balanced acc. at {threshold:.2f}": [pp.scores(y.iloc[b], final_oof.iloc[b], threshold)["balanced_acc"] for _, b in folds],
}, index=pd.Index(range(1, 6), name="window"))
display(window_f1.round(4))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(curve.index, curve.values, color=BLUE)
ax.axvline(threshold, color=ORANGE, linestyle="--", label=f"chosen {threshold:.2f}")
ax.axvline(0.5, color=GREY, linestyle=":", label="default 0.50")
ax.set(xlabel="threshold", ylabel="F1 (pooled windows)", title=f"{final_name}: F1 against the threshold")
ax.legend()
plt.tight_layout()
plt.show()

- Pooled over the five windows, F1 peaks at a threshold of **0.43** (0.7511, against 0.7408 at 0.50).
- The best threshold of single windows ranges from 0.35 to 0.55, so finer tuning would only fit noise: 0.43 improves F1 in three windows and costs a little in the other two.
- At 0.43 the model beats the F1 floor in **every** window, by +0.015 to +0.062.
- A lower threshold means more purchases are labelled "repeat": recall of repeat purchases rises and fewer non-repeat purchases are detected. The cost is visible on the holdout below.

## 4.8 The sealed holdout: one honest estimate


Everything above was chosen on the development windows. The final procedure (tuned pipeline + threshold) is now refitted on **all development rows** and scored **once** on the holdout: the last 12 months of `train.csv`, which no choice has seen. Two references are scored on the same rows: the always-"repeat" floor and an untuned logistic regression with all columns and threshold 0.5, close to the homework approach.

In [ ]:
final_pipeline = tuned_pipeline(final_name).fit(X, y)
holdout_probability = final_pipeline.predict_proba(design.X_hold)[:, 1]

homework_like = pp.make_pipeline(LogisticRegression(max_iter=5000, random_state=R)).fit(X, y)
homework_probability = homework_like.predict_proba(design.X_hold)[:, 1]

holdout = pd.DataFrame({
    f"{final_name} (tuned), threshold {threshold:.2f}": pp.scores(design.y_hold, holdout_probability, threshold),
    f"{final_name} (tuned), threshold 0.50": pp.scores(design.y_hold, holdout_probability, 0.5),
    "LogReg default, all columns, 0.50": pp.scores(design.y_hold, homework_probability, 0.5),
    "always 'repeat'": pp.scores(design.y_hold, np.ones(len(design.y_hold)), 0.5),
}).T.drop(columns="F1_floor")
display(holdout.round(4))
print(f"Holdout repeat rate: {design.y_hold.mean():.3f}; F1 floor: {2 * design.y_hold.mean() / (1 + design.y_hold.mean()):.4f}")
cv_estimate = window_f1[f"F1 at {threshold:.2f}"].mean()
print(f"Validation-window estimate of F1: {cv_estimate:.4f} | holdout F1: {holdout.iloc[0]['F1']:.4f}")
matrix = confusion_matrix(design.y_hold, (holdout_probability >= threshold).astype(int))
display(pd.DataFrame(matrix, index=["actual: no repeat", "actual: repeat"],
                     columns=["predicted: no repeat", "predicted: repeat"]))

- On the 742 holdout purchases the final procedure reaches **ROC-AUC 0.814 and F1 0.794** (threshold 0.43), against a floor of 0.738 and 0.763 for the homework-style logistic regression (untuned, all columns, threshold 0.5).
- The holdout AUC agrees with the validation windows (0.800 ± 0.023): choosing among many candidates on those windows did not overfit them.
- The holdout F1 is higher than the window average (0.750) mostly because the holdout year has more repeat purchases (58.5 % against about 54 %), which raises the floor too: the **margin over the floor is similar** (+0.056 on the holdout, +0.047 on the windows). The same should be expected on Kaggle: the absolute F1 will follow the share of repeats in 2025–2026, the margin over the floor should stay around +0.05.
- The threshold chosen for F1 finds **88 % of repeat purchases but only 54 % of non-repeat purchases** (165 of 308). At 0.50 the model would detect 67 % of non-repeats, with a lower F1 (0.779). For a retention campaign, which targets the non-repeats, a different operating point is better: this is studied in notebook 05.

## 4.9 Final model for deployment


The final model is the same pipeline, with the same settings and threshold, refitted on **all 6,469 usable labelled rows** (development, gap and holdout), so it learns from the most recent purchases before predicting 2025–2026. Its configuration is saved as JSON (read by notebooks 05 and 06) and the fitted pipeline with `joblib` (week 2: save the model *with* its preprocessing).

In [ ]:
config = {
    "model": final_name,
    "best_params": {k: v for k, v in searches[final_name].best_params_.items()},
    "threshold": threshold,
    "feature_selection": {"strategy": "vote (>= 2 of filter, RFE, L1)", "budget": pp.VOTE_BUDGET,
                          "redundancy": pp.VOTE_REDUNDANCY, "l1_C": pp.VOTE_L1_C},
    "validation_windows": {"ROC_AUC_mean": float(tuned_scores[final_name]["ROC_AUC"].mean()),
                           "ROC_AUC_std": float(tuned_scores[final_name]["ROC_AUC"].std()),
                           "F1_at_threshold_mean": float(cv_estimate)},
    "holdout": {k: float(v) for k, v in holdout.iloc[0].items()},
    "training_rows": int(len(design.X_all)),
}
# The final model is rebuilt from the configuration (no temporary cache) and fitted on every usable row.
final_model = pp.final_pipeline(config).fit(design.X_all, design.y_all)
config["selected_columns"] = [str(c) for c in pp.selected_names(final_model)]
joblib.dump(final_model, OUTPUTS / "final_model.joblib")
(OUTPUTS / "final_config.json").write_text(json.dumps(config, indent=2, default=str))
search_log = {name: {"best_params": search.best_params_, "best_ROC_AUC": float(search.best_score_)}
              for name, search in searches.items()}
(OUTPUTS / "search_results.json").write_text(json.dumps(search_log, indent=2, default=str))
print(f"Saved {final_name} trained on {len(design.X_all)} rows; it uses {len(config['selected_columns'])} columns; threshold {threshold:.2f}")
cache_folder.cleanup()